In [2]:
import numpy as np
import xarray as xr

## Load dataset

In [1]:
adcp_filename = "dep0085_socib-rv_scb-rdi001_L1_2026-07-13_data_dt.nc"

In [3]:
nc_adcp = xr.open_dataset(adcp_filename)
nc_adcp

<xarray.Dataset> Size: 10MB
Dimensions:            (time: 789, depth_adcp4: 50)
Coordinates:
  * time               (time) datetime64[ns] 6kB 2026-07-13T07:28:10 ... 2026...
    LAT                (time) float64 6kB ...
    LON                (time) float64 6kB ...
    DEPTH_ADCP4        (depth_adcp4) float64 400B ...
Dimensions without coordinates: depth_adcp4
Data variables: (12/78)
    trajectory         |S22 22B ...
    QC_LAT             (time) float32 3kB ...
    QC_LON             (time) float32 3kB ...
    VEL_NOR            (time, depth_adcp4) float64 316kB ...
    QC_VEL_NOR         (time, depth_adcp4) float32 158kB ...
    VEL_EAS            (time, depth_adcp4) float64 316kB ...
    ...                 ...
    QC_BT_DEPTH_BEAM3  (time) float32 3kB ...
    BT_DEPTH_BEAM4     (time) float64 6kB ...
    QC_BT_DEPTH_BEAM4  (time) float32 3kB ...
    BT_DEPTH_MIN       (time) float64 6kB ...
    QC_BT_DEPTH_MIN    (time) float32 3kB ...
    AMP_BEAM_AVG       (time, depth_adcp4) float64 316kB ...
Attributes: (12/61)
    title:                               Data from instrument SCB-RDi001 on p...
    institution:                         SOCIB (Sistema de Observación y pred...
    netcdf_version:                      3.0
    Conventions:                         CF-1.6
    abstract:                            Data from the ADCP installed at the ...
    summary:                             Data from the ADCP installed at the ...
    ...                                  ...
    instrument_model:                    Ocean Surveyor ADCP 150KHz
    distribution_statement:              Approved for public release. Distrib...
    license:                             Approved for public release. Distrib...
    citation:                            Balearic Island Coastal and Observin...
    acknowledgement:                     Balearic Islands Coastal Observing a...
    update_interval:                     every 0 seconds

In [ ]:
# ============================================================
# PREPARE A TRANSECT
# ============================================================

def prepare_transect(tr):

    i0 = tr["start"]
    i1 = tr["end"]


    idx = np.arange(
        i0,
        i1 + 1
    )


    lon_t = lon[idx].copy()
    lat_t = lat[idx].copy()

    time_t = time[idx].copy()

    u_t = u[idx, :].copy()
    v_t = v[idx, :].copy()

    bottom_t = bottom_full[
        idx
    ].copy()


    # --------------------------------------------------------
    # DETERMINE SHALLOW / DEEP END
    # --------------------------------------------------------

    # Use a small average around each endpoint
    n_edge = min(
        5,
        len(bottom_t)
    )


    start_bottom = np.nanmedian(
        bottom_t[:n_edge]
    )

    end_bottom = np.nanmedian(
        bottom_t[-n_edge:]
    )


    # --------------------------------------------------------
    # REVERSE IF THE TRANSECT STARTS DEEP
    # --------------------------------------------------------

    if (
        np.isfinite(start_bottom)
        and
        np.isfinite(end_bottom)
        and
        start_bottom > end_bottom
    ):

        lon_t = lon_t[::-1]
        lat_t = lat_t[::-1]
        time_t = time_t[::-1]

        u_t = u_t[::-1, :]
        v_t = v_t[::-1, :]

        bottom_t = bottom_t[::-1]


    # --------------------------------------------------------
    # DISTANCE FROM SHALLOW END
    # --------------------------------------------------------

    distance = cumulative_distance_km(
        lon_t,
        lat_t
    )


    # --------------------------------------------------------
    # TRANSECT UNIT VECTOR
    #
    # shallow -> deep
    # --------------------------------------------------------

    lat_mean = np.nanmean(
        lat_t
    )


    dx = (
        lon_t[-1] - lon_t[0]
    ) * np.cos(
        np.deg2rad(lat_mean)
    )


    dy = (
        lat_t[-1] - lat_t[0]
    )


    norm_direction = np.sqrt(
        dx**2 + dy**2
    )


    if norm_direction == 0:

        raise ValueError(
            "Transect has zero direction."
        )


    east_unit = (
        dx / norm_direction
    )

    north_unit = (
        dy / norm_direction
    )


    # --------------------------------------------------------
    # ROTATE VELOCITY
    #
    # Positive = toward deep end
    # Negative = toward shallow end
    # --------------------------------------------------------

    vel_sd = (
        u_t * east_unit
        +
        v_t * north_unit
    )


    return {
        "lon": lon_t,
        "lat": lat_t,
        "time": time_t,
        "distance": distance,
        "bottom": bottom_t,
        "velocity": vel_sd,
        "start_bottom": bottom_t[0],
        "end_bottom": bottom_t[-1]
    }


# ============================================================
# PREPARE ALL TRANSECTS
# ============================================================

prepared = [
    prepare_transect(tr)
    for tr in transects
]